In [11]:
import numpy as np
import pandas as pd

np.random.seed(42)

# =========================================
# Load Citizen Trust Dataset
# =========================================

citizens = pd.read_csv(
    "F:/GraduationDataset/1-Citizen Trust Score Model/citizen_trust_dataset.csv"
)

# لو مفيش trust_score استخدم valid_ratio مؤقتًا
if "trust_score" not in citizens.columns:
    citizens["trust_score"] = citizens["valid_ratio"]

print("✅ Citizen Dataset Loaded")

# =========================================
# Incident Types & Area Types
# =========================================

incident_types = [
    "Electricity",
    "Water",
    "Road",
    "Gas",
    "Waste"
]

area_types = [
    "Residential",
    "Commercial",
    "Industrial"
]

# =========================================
# Incident Type Risk Mapping
# =========================================

type_risk = {
    "Gas": 0.95,
    "Electricity": 0.80,
    "Road": 0.65,
    "Water": 0.50,
    "Waste": 0.35
}

# =========================================
# Severity Label Function
# =========================================

def generate_severity_label(score):

    if score >= 0.70:
        return "Critical"

    elif score >= 0.55:
        return "High"

    elif score >= 0.35:
        return "Medium"

    else:
        return "Low"

# =========================================
# Generate Incidents
# =========================================

data = []

for incident_id in range(1, 30001):

    # Random Citizen
    citizen = citizens.sample(1).iloc[0]

    trust = citizen["trust_score"]

    # Incident Info
    incident_type = np.random.choice(incident_types)
    area_type = np.random.choice(area_types)

    hour = np.random.randint(0, 24)
    day_of_week = np.random.randint(0, 7)

    # AI Related Features
    anomaly_score = round(
        np.random.beta(2, 5),
        2
    )

    image_auth_score = round(
        np.random.beta(5, 2),
        2
    )

    reports_nearby_1h = np.random.poisson(lam=3)

    historical_severity_avg = round(
        np.random.uniform(1.5, 3.5),
        2
    )

    # =========================================
    # Incident Type Risk
    # =========================================

    incident_risk = type_risk[incident_type]

    # =========================================
    # Severity Logic
    # =========================================

    severity_numeric = (
        0.30 * (1 - trust) +
        0.25 * anomaly_score +
        0.20 * (1 - image_auth_score) +
        0.15 * min(reports_nearby_1h / 10, 1) +
        0.10 * incident_risk
    )

    # Random Noise
    severity_numeric += np.random.normal(0, 0.05)

    # Clip Between 0 and 1
    severity_numeric = np.clip(
        severity_numeric,
        0,
        1
    )

    # Severity Label
    severity_label = generate_severity_label(
        severity_numeric
    )

    # =========================================
    # Save Record
    # =========================================

    data.append({
        "incident_id": incident_id,

        "citizen_id": citizen["citizen_id"],

        "incident_type": incident_type,

        "area_type": area_type,

        "hour": hour,

        "day_of_week": day_of_week,

        "citizen_trust_score": round(
            trust,
            2
        ),

        "anomaly_score": anomaly_score,

        "image_authenticity_score": image_auth_score,

        "reports_nearby_1h": reports_nearby_1h,

        "historical_severity_avg": historical_severity_avg,

        "incident_type_risk": incident_risk,

        "severity_score": round(
            severity_numeric,
            3
        ),

        "severity_label": severity_label
    })

# =========================================
# Create DataFrame
# =========================================

df_incidents = pd.DataFrame(data)

# =========================================
# Save Dataset
# =========================================

df_incidents.to_csv(
    "incident_severity_dataset_v2.csv",
    index=False
)

# =========================================
# Dataset Info
# =========================================

print("\n✅ Incident Dataset Generated")
print(df_incidents.shape)

print("\n========================")
print(df_incidents.head())

print("\n========================")
print("Severity Distribution")
print("========================\n")

print(
    df_incidents["severity_label"].value_counts()
)

✅ Citizen Dataset Loaded

✅ Incident Dataset Generated
(30000, 14)

   incident_id  citizen_id incident_type    area_type  hour  day_of_week  \
0            1     10651.0         Waste   Industrial     1            0   
1            2      7610.0   Electricity   Commercial    15            2   
2            3      3210.0         Waste   Industrial    10            3   
3            4     14817.0         Waste  Residential    19            1   
4            5     11189.0         Waste   Industrial    14            5   

   citizen_trust_score  anomaly_score  image_authenticity_score  \
0                 0.70           0.32                      0.56   
1                 0.73           0.47                      0.64   
2                 0.87           0.42                      0.40   
3                 0.11           0.47                      0.52   
4                 0.60           0.11                      0.69   

   reports_nearby_1h  historical_severity_avg  incident_type_risk  \
0  

In [12]:
df = pd.read_csv("D:/SALEM_AI/Machine Learning Models/2-Incident Severity Prediction/Dataset/incident_severity_dataset_v2.csv")


In [13]:
print(df["severity_label"].value_counts())


severity_label
Medium      14843
Low         13048
High         2037
Critical       72
Name: count, dtype: int64
